In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../scripts')

In [3]:
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import os
import seaborn as sns
from tqdm import tqdm
import torch
import mintflow

In [4]:
from utils import set_seed
from configs.adata_crc_config import ADATA_ARGS as ADATA_CRC_ARGS
from configs.adata_merfish_config import ADATA_ARGS as ADATA_MERFISH_ARGS
from counterfactual_analysis import compute_lfc_metrics, compute_rmse, compute_edistance, mixing_index


set_seed(0)
DATASET_NAME = "merfish"  # Options: "crc", "merfish"

CRC_BASE_PATH = "/data2/a330d/datasets/crc/raw_zenodo"
CRC_SLIDES = ['crc_232' , 'crc_231', 'crc_221', 'crc_210', 'crc_242', 'crc_120']
CRC_CELLTYPES = [
    "Endothelial",
    "Epithelial",
    "Fibroblast",
    "Myeloid",
    "T_cell",
]

MERFISH_BASE_PATH = "/data/a330d/datasets/MERFISH_mouse_brain"
MERFISH_SLIDES = ['C57BL6J-2.036', 'C57BL6J-2.039', 'C57BL6J-2.041']
MERFISH_CELLTYPES = [
    'glutamatergic neuron',
    'oligodendrocyte',
    'astrocyte',
    'GABAergic neuron',
    'endothelial cell',
]

ADATA_BASE_PATH = CRC_BASE_PATH if DATASET_NAME == "crc" else MERFISH_BASE_PATH
SLIDES = CRC_SLIDES if DATASET_NAME == "crc" else MERFISH_SLIDES
DATA_ARGS = ADATA_CRC_ARGS if DATASET_NAME == "crc" else ADATA_MERFISH_ARGS
HOLDOUT_CELLTYPES = CRC_CELLTYPES if DATASET_NAME == "crc" else MERFISH_CELLTYPES

LABELS_KEY = DATA_ARGS.get('labels_key')
DOMAINS_KEY = DATA_ARGS.get('domains_key')
N_TOP_GENES = DATA_ARGS.get('n_top_genes')
PATIENT_ID = DATA_ARGS.get('batch_key')
CONTROL_DOMAIN = DATA_ARGS.get('control_domains')[0]  # Assuming only one control domain for simplicity
HOLDOUT_DOMAINS = DATA_ARGS.get('holdout_domains')
N_NEIGHBORS = 5
USE_WANDB = 'False'
MODEL_OUTPUT_PATH = "/data/a330d/data/ood/trained"
ADATA_SAVE_PATH = f"/data/a330d/datasets/{DATASET_NAME}/processed"
X_POS = 'CenterX_global_px'
Y_POS = 'CenterY_global_px'

In [5]:
# For crc set domain_key to 'typ' - models were trained before 'typ_clean' was introduced - doesn't change downstream analysis
DOMAINS_KEY = 'typ' if DATASET_NAME == "crc" else DOMAINS_KEY

In [37]:
slide_id = SLIDES[2]
slide_id

'C57BL6J-2.041'

In [38]:
top_n = 50 # Number of DE genes to use for evaluation
chunk_size = 90000
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_recon = False

In [39]:
path_output_files = f"{MODEL_OUTPUT_PATH}/{slide_id}/mintflow/"

In [40]:
# Search os.path.join(path_output_files) directory, and load highest epoch checkpoint
checkpoint_files = [f for f in os.listdir(path_output_files) if f.startswith("checkpoint_epoch_")]
checkpoint_epochs = [int(f.split("checkpoint_epoch_")[1].split(".pt")[0]) for f in checkpoint_files]
latest_epoch = max(checkpoint_epochs)
checkpoint_path = os.path.join(path_output_files, f"checkpoint_epoch_{latest_epoch}.pt")
checkpoint_path

'/data/a330d/data/ood/trained/C57BL6J-2.041/mintflow/checkpoint_epoch_30.pt'

In [41]:
checkpoint_mintflow = torch.load(
    checkpoint_path,
    map_location='cpu',
    weights_only=False
)
checkpoint_mintflow['model'].to(device)
print("Loaded the checkpoint.")

Loaded the checkpoint.


In [42]:
torch.cuda.empty_cache()

In [43]:
adata = sc.read(
    f"{ADATA_SAVE_PATH}/{slide_id}.h5ad"
)

In [44]:
kwargs_neighbourhood_graph = {
    'spatial_key': 'spatial',
    'library_key': None,
    'set_diag': False,
    'delaunay': False,
    'n_neighs': 5
}
adata.uns = {}

import squidpy as sq
sq.gr.spatial_neighbors(
    adata=adata,
    **kwargs_neighbourhood_graph
)

INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        


In [45]:
num_generated_realisations = 3
n_cells = adata.n_obs
n_genes = adata.n_vars

recon_x = np.zeros((n_cells, n_genes), dtype=np.float32)

graph = adata.obsp["spatial_connectivities"]

indices = np.arange(adata.n_obs)

for start in tqdm(range(0, len(indices), chunk_size), desc="Generating in-silico data in batches"):
    batch_idx = indices[start:start + chunk_size]

    # include neighbors
    neighbor_idx = graph[batch_idx].nonzero()[1]
    all_idx = np.unique(np.concatenate([batch_idx, neighbor_idx]))

    adata_batch = adata[all_idx].copy()

    with torch.no_grad():
        res = mintflow.generate_insilico_ST_data(
            adata=adata_batch,
            obskey_celltype=LABELS_KEY,
            obspkey_neighbourhood_graph="spatial_connectivities",
            device=device,
            batch_index_trainingdata=0,
            num_generated_realisations=num_generated_realisations,
            model=checkpoint_mintflow["model"],
            data_mintflow=checkpoint_mintflow["data_mintflow"],
            dict_all4_configs=checkpoint_mintflow["dict_all4_configs"],
            estimate_spatial_sizefactors_on_sections=[0]
        )

    # compute mean reconstruction across realizations
    recon_batch = np.stack([
        r["MintFlow_Generated_Xint"] + r["MintFLow_Generated_Xmic"]
        for r in res["list_generated_realisations_ie_expressions"]
    ]).mean(0)

    # map results back to global indices
    pos = np.searchsorted(all_idx, batch_idx)

    recon_x[batch_idx] = recon_batch[pos]

    torch.cuda.empty_cache()

Generating in-silico data in batches:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating on tissue section: 0:   0%|          | 0/9 [00:00<?, ?it/s]

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


Generating the realisations of the expression data (i.e. generative samples) for the provided in silico tissue…

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:546: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_index),
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
Generating in-silico data in batches: 100%|██████████| 1/1 [00:27<00:00, 27.06s/it]


In [46]:
adata.obsm['recon_x'] = recon_x

# Predict for all cell types

In [ ]:
target_celltypes = HOLDOUT_CELLTYPES
results = []

for target_ct in tqdm(target_celltypes, desc="Target Cell Types"):
    is_holdout_ct = adata.obs[LABELS_KEY] == target_ct
    is_in_control_domain = adata.obs[DOMAINS_KEY].str.contains(CONTROL_DOMAIN)
    control_mask = is_holdout_ct & is_in_control_domain
    control_idx = np.where(control_mask)[0]
    control = adata.layers['counts'][control_mask.values, :].todense()
    control = np.asarray(control)
    if use_recon:
        control = adata.obsm['recon_x'][control_mask.values, :]

    for hd in HOLDOUT_DOMAINS:
        is_in_holdout_domain = adata.obs[DOMAINS_KEY].str.contains(hd)
        target_mask = is_holdout_ct & is_in_holdout_domain
        target_idx = np.where(target_mask)[0]
        perturbed_x = np.zeros((adata.n_obs, adata.n_vars), dtype=np.float32)

        for start in tqdm(range(0, len(target_idx), chunk_size), desc="Generating in-silico data in batches"):
            batch_cells = target_idx[start:start + chunk_size]

            # include neighbors
            neighbor_idx = graph[batch_cells].nonzero()[1]
            all_idx = np.unique(np.concatenate([batch_cells, neighbor_idx]))

            adata_batch = adata[all_idx].copy()
            adata_batch.uns = {}

            # map global indices → batch indices
            batch_pos = np.searchsorted(all_idx, batch_cells)

            # choose control replacements
            sampled_control = np.random.choice(control_idx, size=len(batch_cells), replace=True)

            # replace gene expression for target cells
            adata_batch.X[batch_pos] = adata.X[sampled_control]

            with torch.no_grad():
                res = mintflow.generate_insilico_ST_data(
                    adata=adata_batch,
                    obskey_celltype=LABELS_KEY,
                    obspkey_neighbourhood_graph="spatial_connectivities",
                    device=device,
                    batch_index_trainingdata=0,
                    num_generated_realisations=num_generated_realisations,
                    model=checkpoint_mintflow["model"],
                    data_mintflow=checkpoint_mintflow["data_mintflow"],
                    dict_all4_configs=checkpoint_mintflow["dict_all4_configs"],
                    estimate_spatial_sizefactors_on_sections=[0]
                )

            recon_batch = np.stack([
                r["MintFlow_Generated_Xint"] + r["MintFLow_Generated_Xmic"]
                for r in res["list_generated_realisations_ie_expressions"]
            ]).mean(0)

            perturbed_x[batch_cells] = recon_batch[batch_pos]

            torch.cuda.empty_cache()

        # Compute metrics
        counterfactual = perturbed_x[target_mask]
        target = adata.layers['counts'][target_mask.values, :].todense()
        target = np.asarray(target)
        if use_recon:
            target = adata.obsm['recon_x'][target_mask.values, :]

        mask_holdout = (adata.obs[LABELS_KEY] == target_ct) & (adata.obs[DOMAINS_KEY].isin(HOLDOUT_DOMAINS))
        adata.obs['is_holdout'] = mask_holdout
        
        pear, spear, prec, dir_match, deg = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=top_n)
        rmse = compute_rmse(observed=target, predicted=counterfactual, deg=deg)
        edist_global = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=1e4)
        edist_local = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=1e4, local=True)
        edist_pca_log = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=1e4, local=True, use_pca=True)
        edist_pca = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=1e4, local=True, use_pca=True, log1p=False)
        mix_idx = mixing_index(observed=target, predicted=counterfactual, library_size=1e4)
        _, _, _, dir_match_k, _ = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=top_n, direction_match_normalize="k")
        _, _, _, dir_match_gt, _ = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=top_n, direction_match_normalize="gt_topk")

        results.append(
            dict(
                dataset_name=DATASET_NAME,
                sid=slide_id,
                control_domain=CONTROL_DOMAIN,
                target_domain=hd,
                n_deg=top_n,
                model_name="mintflow",
                holdout_celltype=target_ct,
                spearman=spear,
                pearson=pear,
                precision=prec,
                direction_match=dir_match,
                direction_match_k=dir_match_k,
                direction_match_gt=dir_match_gt,
                mixing_index=mix_idx,
                edistance_global=edist_global,
                edistance_local=edist_local,
                edistance_pca_log=edist_pca_log,
                edistance_pca=edist_pca,
                rmse=np.log10(rmse),
            )
        )

Target Cell Types:   0%|          | 0/5 [00:00<?, ?it/s]/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/scipy/sparse/_index.py:201: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray_sparse(i, j, x)


Evaluating on tissue section: 0:   0%|          | 0/9 [00:00<?, ?it/s]

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


Generating the realisations of the expression data (i.e. generative samples) for the provided in silico tissue…

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:546: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_index),
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
Generating in-silico data in batches: 100%|██████████| 1/1 [00:16<00:00, 16.32s/it]
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/scipy/sparse/_index.py:201: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_array

...

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:546: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_index),
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
Generating in-silico data in batches: 100%|██████████| 1/1 [00:14<00:00, 14.03s/it]
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/scipy/sparse/_index.py:201: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_array

Evaluating on tissue section: 0:   0%|          | 0/9 [00:00<?, ?it/s]

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


Generating the realisations of the expression data (i.e. generative samples) for the provided in silico tissue…

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:546: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_index),
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
Target Cell Types: 100%|██████████| 5/5 [03:21<00:00, 40.35s/it]


In [48]:
df_results = pd.DataFrame(results)
df_csv_fname = f"mintflow_{slide_id}.csv"
os.makedirs(os.path.join(f'../results/mintflow_{DATASET_NAME}'), exist_ok=True)
df_results.to_csv(os.path.join(f'../results/mintflow_{DATASET_NAME}', df_csv_fname), index=False)

In [49]:
df_results

,dataset_name,sid,control_domain,target_domain,n_deg,model_name,holdout_celltype,spearman,pearson,precision,direction_match,direction_match_k,dir_match_gt,mixing_index,edistance_global,edistance_local,edist_pca_log,edist_pca,rmse
0,merfish,C57BL6J-2.041,Thalamus,Isocortex,50,mintflow,glutamatergic neuron,0.855750,0.970033,0.40,1.000000,0.40,0.96,0.046592,56.786398,56.763433,18.851873,771.068835,4.415111
1,merfish,C57BL6J-2.041,Thalamus,Fiber_tracts,50,mintflow,glutamatergic neuron,0.812629,0.919151,0.32,1.000000,0.32,0.96,0.507898,56.089785,58.619808,18.959404,853.807245,3.437548
2,merfish,C57BL6J-2.041,Thalamus,Isocortex,50,mintflow,oligodendrocyte,0.222569,0.496704,0.14,0.714286,0.10,0.20,0.208103,69.579885,68.098275,14.794299,1961.416345,3.366832
3,merfish,C57BL6J-2.041,Thalamus,Fiber_tracts,50,mintflow,oligodendrocyte,0.066315,0.719428,0.14,0.714286,0.10,0.36,0.075181,75.832495,75.980936,19.164628,3174.220190,3.952712
4,merfish,C57BL6J-2.041,Thalamus,Isocortex,50,mintflow,astrocyte,0.819832,0.813064,0.22,1.000000,0.22,0.74,0.919427,66.966125,64.727936,19.863591,1764.113098,3.647163
5,merfish,C57BL6J-2.041,Thalamus,Fiber_tracts,50,mintflow,astrocyte,0.450660,0.505228,0.16,0.750000,0.12,0.62,0.653003,67.510985,66.547187,20.236565,1883.922375,3.629495
6,merfish,C57BL6J-2.041,Thalamus,Isocortex,50,mintflow,GABAergic neuron,0.856327,0.954316,0.36,1.000000,0.36,0.98,0.176842,58.852342,58.844337,19.819591,1137.283044,3.559943
7,merfish,C57BL6J-2.041,Thalamus,Fiber_tracts,50,mintflow,GABAergic neuron,0.639184,0.712801,0.32,0.812500,0.26,0.48,0.403141,59.374626,60.345570,17.638187,1053.215088,2.859374
8,merfish,C57BL6J-2.041,Thalamus,Isocortex,50,mintflow,endothelial cell,0.897239,0.938193,0.72,1.000000,0.72,0.84,0.143744,66.864108,65.592385,17.957679,1904.760425,3.649237
9,merfish,C57BL6J-2.041,Thalamus,Fiber_tracts,50,mintflow,endothelial cell,0.367779,0.989145,0.12,1.000000,0.12,0.20,0.141994,69.077723,67.712296,19.310896,2216.510840,3.190697
